In [ ]:
# Install required packages
!pip install ultralytics -q

In [ ]:
# Import and verify ultralytics version
import ultralytics

print(f"ultralytics version: {ultralytics.__version__}")

In [ ]:
# Dataset configuration
dataset_path = "PATH_TO_YOUR_DATASET"  # Update this with your dataset path

# Define paths for training
train_label_dir = f"{dataset_path}/labels/train"
val_label_dir = f"{dataset_path}/labels/val"
yaml_path = f"{dataset_path}/data.yaml"

In [ ]:
import os

def fix_labels(label_dir, max_class_id=0):
    """
    Fix class IDs in YOLO format label files by ensuring they don't exceed max_class_id.
    
    Args:
        label_dir (str): Directory containing label files
        max_class_id (int): Maximum allowed class ID (default: 0)
    """
    for label_file in os.listdir(label_dir):
        label_path = os.path.join(label_dir, label_file)
        updated_lines = []

        with open(label_path, "r") as file:
            lines = file.readlines()
            for line in lines:
                parts = line.strip().split()
                class_id = int(parts[0])
                
                if class_id > max_class_id:
                    print(f"Invalid class ID {class_id} in {label_path}")
                    class_id = max_class_id
                
                parts[0] = str(class_id)
                updated_lines.append(" ".join(parts))

        with open(label_path, "w") as file:
            file.write("\n".join(updated_lines) + "\n")

# Process train and validation labels
fix_labels(train_label_dir)
fix_labels(val_label_dir)

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 model
model = YOLO("yolov8n.pt")  # Choose model size based on your hardware capabilities

# Define save directory for model checkpoints
save_dir = "PATH_TO_SAVE_DIR"  # Update this with your save directory path

# Train the model
model.train(
    data=yaml_path,                     # Path to data configuration file
    epochs=100,                         # Number of training epochs
    batch=32,                          # Batch size
    imgsz=640,                         # Image size
    optimizer="Adam",                   # Optimizer
    lr0=0.001,                         # Initial learning rate
    weight_decay=0.0005,               # Regularization term
    patience=10,                        # Early stopping patience
    workers=4,                         # Number of workers for data loading
    project=save_dir,                  # Project directory
    name="nutritional_label_detector", # Experiment name
    device=0,                          # GPU device (0 for first GPU)
    cos_lr=True,                       # Cosine learning rate scheduler
    save_period=10                     # Save checkpoint frequency
)

In [ ]:
import os

# Ensure the destination directory exists
destination_dir = "PATH_TO_SAVED_MODEL"  # Update this with your model save directory
os.makedirs(destination_dir, exist_ok=True)

In [ ]:
import shutil

# Define paths for model weights
source_path = "PATH_TO_TRAINED_MODEL/best.pt"  # Path to your trained model weights
destination_path = os.path.join(destination_dir, "best.pt")  # Using previously defined destination_dir

# Copy model weights to destination
shutil.copy(source_path, destination_path)
print(f"Model weights saved to {destination_path}")

In [ ]:
# Verify model file existence
if os.path.exists(source_path):
    print(f"Model weights found: {source_path}")
else:
    raise FileNotFoundError(f"Model weights not found at: {source_path}")

In [ ]:
from ultralytics import YOLO

# Load the trained model
model_path = "PATH_TO_SAVED_MODEL/best.pt"  # Update with your model path
model = YOLO(model_path)

In [ ]:
import os

# Directory containing test images
image_dir = "PATH_TO_TEST_IMAGES"  # Update with your test images directory path

# Get image paths (supports jpg and png formats)
image_paths = [
    os.path.join(image_dir, fname) 
    for fname in os.listdir(image_dir) 
    if fname.lower().endswith(('.jpg', '.png'))
][:20]  # Limit to first 20 images

In [ ]:
# Perform inference on test images
for img_path in image_paths:
    # Run inference
    results = model.predict(img_path)

    # Process and visualize results
    for result in results:
        # Display predictions
        result.show()  
        
        # Save annotated images
        save_dir = "PATH_TO_OUTPUT_DIR"  # Update with your desired output directory
        result.save(save_dir)  

In [ ]:
import os
import shutil

def save_project(source_dir="PATH_TO_SOURCE_DIR", project_name="nutrition_table_detection"):
    """
    Save and zip project directory
    
    Args:
        source_dir (str): Source directory path
        project_name (str): Name of the project/output zip file
    """
    # Create zip archive
    output_path = os.path.join(os.getcwd(), project_name)
    if os.path.exists(source_dir):
        shutil.make_archive(output_path, 'zip', source_dir)
        print(f"Project saved as: {output_path}.zip")
    else:
        print(f"Source directory not found: {source_dir}")

# Usage
save_project("PATH_TO_SOURCE_DIR")  # Update with your source directory path